# Lesson 1.1 - Programming Basics for AI Engineering

## Objectives
- Use Python types, control flow, functions, and modules confidently.
- Apply input validation, exceptions, and logging.
- Build mini scripts similar to AI data-team workflows.


## Environment & Imports
Install with `uv pip install -r requirements.txt`.


In [ ]:
from pathlib import Path
import logging
import statistics
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("lesson_1_1")


## Section A - Variables, Types, and Control Flow


In [ ]:
age = 29
score = 92.5
name = "Amina"
is_active = True

print(type(age), type(score), type(name), type(is_active))

scores = [88, 76, 92, 95, 61]
passed = [s for s in scores if s >= 70]
print("Passed:", passed)

if len(passed) >= 4:
    print("Quality gate: pass")
else:
    print("Quality gate: review")


## Section B - Functions and Reuse


In [ ]:
def compute_bmi(weight_kg: float, height_m: float) -> float:
    if weight_kg <= 0 or height_m <= 0:
        raise ValueError("weight_kg and height_m must be positive")
    return weight_kg / (height_m ** 2)


def normalize_scores(values: list[float]) -> list[float]:
    if not values:
        raise ValueError("values cannot be empty")
    lo, hi = min(values), max(values)
    if lo == hi:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]

print("BMI:", round(compute_bmi(78, 1.8), 2))
print("Normalized:", normalize_scores(scores))


## Section C - Exceptions and Logging


In [ ]:
class InvalidInputError(ValueError):
    pass


def parse_positive_float(raw: str, field_name: str) -> float:
    try:
        value = float(raw)
    except ValueError as exc:
        raise InvalidInputError(f"{field_name} must be numeric") from exc
    if value <= 0:
        raise InvalidInputError(f"{field_name} must be > 0")
    return value

for raw in ["12.1", "abc", "-1"]:
    try:
        parsed = parse_positive_float(raw, "monthly_income")
        logger.info("Parsed value=%s", parsed)
    except InvalidInputError as exc:
        logger.warning("Validation failed: %s", exc)


## Section D - Mini ETL Script Example


In [ ]:
IRIS_URL = "https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv"
OUT_DIR = Path("../data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_cols = {"sepal_length", "sepal_width", "petal_length", "petal_width", "species"}

df = pd.read_csv(IRIS_URL)
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing columns: {sorted(missing)}")

df["petal_area_proxy"] = df["petal_length"] * df["petal_width"]
summary = (
    df.groupby("species", as_index=False)
      .agg(mean_sepal_length=("sepal_length", "mean"), mean_petal_area=("petal_area_proxy", "mean"), n=("species", "size"))
)
out_path = OUT_DIR / "iris_summary.csv"
summary.to_csv(out_path, index=False)
print("Saved:", out_path)
display(summary)


In [ ]:
def summarize_numeric(values: list[float]) -> dict[str, float]:
    if not values:
        raise ValueError("values must not be empty")
    return {
        "mean": statistics.mean(values),
        "std": statistics.pstdev(values),
        "min": min(values),
        "max": max(values),
    }

print(summarize_numeric([2, 5, 8, 11]))


## Business Case Studies & Exceptions
- Feature ingestion API receives malformed payloads.
  - Validate schema/ranges early, raise typed errors, log request IDs.
- Nightly retraining silently failed because broad exception swallowed parse errors.
  - Use narrow exception handling and fail-fast on contract violations.
- ETL partial failure strategy: quarantine bad rows, continue valid rows, alert if bad-row ratio crosses threshold.


## Interview Questions & Answers
1. **Q:** Why use Python in AI engineering?  
   **A:** Rapid iteration and rich ML/data ecosystem.
2. **Q:** Syntax error vs runtime exception?  
   **A:** Syntax error blocks parse; runtime exception happens during execution.
3. **Q:** Why type hints?  
   **A:** Better readability and static analysis.
4. **Q:** Why logging over print?  
   **A:** Structured observability with levels and metadata.
5. **Q:** When raise custom exceptions?  
   **A:** Domain-specific validation and clearer error semantics.
6. **Q:** How keep notebook reproducible?  
   **A:** Ordered cells, restart+run-all, fixed seeds.
7. **Q:** Why separate I/O from logic?  
   **A:** Easier tests and reuse.
8. **Q:** Common bug source in ETL?  
   **A:** Schema drift.
9. **Q:** What is defensive programming?  
   **A:** Validate assumptions and reject invalid states.
10. **Q:** How handle partial batch failures?  
    **A:** Continue valid rows and log/monitor rejected rows.
